# Grad-CAM: so sánh ba arm RQ1

Notebook chạy ba model cuối (patience 20, seed 42) trên cùng một bộ ảnh:

- Custom CNN
- MobileNetV3_small pretrained
- MobileNetV3_small scratch

Mỗi class chọn 2 ảnh: một ảnh cả ba model đúng và một ảnh lỗi tiêu biểu.
Nguồn ảnh gồm **test in-domain** và **external GD**.

- Logic chọn ảnh và vẽ nằm trong `src/training/gradcam_compare.py`.
- Grad-CAM dùng `src/training/gradcam.py`.
- GD chỉ dùng phân tích định tính sau đánh giá, không dùng chọn model.


In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import Image, display

ROOT = next(
    path for path in [Path.cwd(), *Path.cwd().parents] if (path / "pyproject.toml").is_file()
)
sys.path.insert(0, str(ROOT))

from src.training.gradcam_compare import run_comparison
from src.utils.paths import experiments_root

RUNS = {
    "Custom CNN": experiments_root() / "rq1" / "rq1_customcnn_p20_s42",
    "MobileNet pretrained": experiments_root() / "rq1" / "rq1_mobilenetv3s_full_p20_s42",
    "MobileNet scratch": experiments_root() / "rq1" / "rq1_mobilenetv3s_scratch_p20_s42",
}
OUTPUT = ROOT / "docs" / "RQ1" / "rq1_analysis" / "gradcam_compare"
print({label: path.is_dir() for label, path in RUNS.items()})


## Sinh Grad-CAM cho test in-domain và external GD

Mỗi nguồn sinh:

- `summary.png`: 3 ảnh lỗi tiêu biểu;
- `<class>.png`: 2 ảnh/class (đúng + lỗi);
- `selection_<source>.csv`: danh sách ảnh đã chọn.


In [ ]:
results = run_comparison(
    RUNS,
    sources=("test", "gd"),
    output_dir=OUTPUT,
    per_class=2,
)
results.keys()


## Test in-domain

In [ ]:
display(Image(filename=results["test"]["summary"]))


## External GD

In [ ]:
display(Image(filename=results["gd"]["summary"]))


## Một ví dụ theo class

Xem thêm các file `<class>.png` trong thư mục kết quả.


In [ ]:
display(Image(filename=results["test"]["classes"]["metal"]))


In [ ]:
display(Image(filename=results["gd"]["classes"]["paper"]))


## Danh sách ảnh đã chọn

Quy tắc:

- **đúng**: cả ba model dự đoán đúng, chọn confidence trung bình cao nhất;
- **lỗi**: ưu tiên ảnh nhiều model cùng sai, sau đó confidence trung bình cao.


In [ ]:
for source in ("test", "gd"):
    print(source)
    display(results[source]["selected"])


## Nhận xét

- Custom CNN thường bám vào biên vật thể và một phần nền.
- MobileNet pretrained tập trung hơn vào cấu trúc/vùng vật thể.
- MobileNet scratch cho heatmap khuếch tán hơn, nhiều vùng ngoài vật thể.
- Trên GD, cả ba model đều lệch domain; pretrained vẫn giữ focus tốt hơn.
- Grad-CAM là phân tích định tính, không chứng minh quan hệ nhân quả.
